In [ ]:
import os
import matplotlib as mpl
"""
if os.environ.get('DISPLAY','') == '':
    print('no display found. Using non-interactive Agg backend')
    mpl.use('Agg')
"""
import xlrd
#import xlwt
import xlsxwriter
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from matplotlib.path import Path
import numpy as np
import seawater as gsw
import scipy.optimize as opt
import pickle
import scipy.io as sio
import pandas as pd #LA
import openpyxl #LA
import xarray as xr #LA

In [ ]:
#def solve_OMP(T,S,NO,Si,T0,S0,NO0,Si0,indS,indWM, ww = [100,10,10,2,1]):
def solve_OMP(T,S,P,T0,S0,P0,indS,indWM, ww = [100,10,10,2]): #weights for mass, temp, sal, preformed phosphate 

    #ww = np.array(ww)**2
    N = len(indS)
    NWM = len(indWM)
    
    T = T[indS]
    S = S[indS]
    P = P[indS]

    mT = np.mean(T)
    rT = Crange(T)
    mS = np.mean(S)
    rS = Crange(S) 
    mP = np.mean(P)
    rP = Crange(P) 
    
    M = ww[0]*np.ones(T.shape)
    T = ww[1]*(T - mT)/rT   
    S = ww[2]*(S - mS)/rS
    P = ww[3]*(P - mP)/rP 

    T0 = T0[indWM]
    S0 = S0[indWM]
    P0 = P0[indWM]

    M0 = ww[0]*np.ones(T0.shape)
    T0 = ww[1]*(T0 - mT)/rT   
    S0 = ww[2]*(S0 - mS)/rS
    P0 = ww[3]*(P0 - mP)/rP

    B = np.vstack( (M0,T0, S0, P0) )

    X = np.zeros((N,NWM))
    for i in range(N):
        y =  [M[i],T[i],S[i], P[i]]
        #X=np.linalg.lstsq(B,y) #
        x=opt.nnls(B,y)
        X[i,:] = x[0]

   
    return X

def Crange(x):
    R = np.max(x)-np.min(x)
    return R

def realocates_values(y,Y,I,J):
    i=0
    for ii in I:
        j=0
        for jj in J:        
            Y[ii,jj] = y[i,j]
            j+=1
        i+=1
    return Y

In [ ]:
def does_OMP(pres,PT,S,P,WTfile):

    wbWT =  pd.read_excel(WTfile)
    #wbWT = openpyxl.load_workbook(WTfile)
    WT = wbWT['NAME'].tolist()
    WT_S = np.array(wbWT['S'].tolist())
    WT_PT = np.array(wbWT['T'].tolist())
    WT_P = np.array(wbWT['Ppre'].tolist()) 
    WT_dens = gsw.dens(WT_S,WT_PT,0)-1000
    
    NP = len(PT)
    NWM = len(WT)
    
    iv2 = np.arange(NP)
    #stU = np.unique(st) #estacions unicas 
    
    TScoords =  []
    for i in range(len(PT)):
        TScoords.append( (S[i],PT[i]) )
    
    ###############
    #MIXING FIGURES
    ################
     
    corrT = np.array([0.,0.,0.,0.,0.,0.,0.,0.,0.,0,0.,0.]) 
    corrS = np.array([0.0,0.0,0.,0.,0.,0.,0.,0.,0.,0.0,0.,0.]) 
    
    #LA - polygon 1
    jpoly_1 = [4, 11, 10]
    vpoly_1 = [(WT_S[i],WT_PT[i]) for i in jpoly_1]
    vpoly_1c = [(WT_S[i]+corrS[i],WT_PT[i]+corrT[i]) for i in jpoly_1]
    #vpoly_1c = [(WT_S[i],WT_PT[i]) for i in jpoly_1]
    ppoly_1c = Path(vpoly_1c)
    #ipoly_1 = np.where((ppoly_1c.contains_points(TScoords)) & (pres>=100.))[0]
    ipoly_1 = np.where((ppoly_1c.contains_points(TScoords)))[0] #INCLUDING SURFACE VALUES, NOT JUST LOWER THAN 100M

    
    #LA - polygon 2
    jpoly_2 = [3, 4, 10]
    vpoly_2 = [(WT_S[i],WT_PT[i]) for i in jpoly_2]
    vpoly_2c = [(WT_S[i]+corrS[i],WT_PT[i]+corrT[i]) for i in jpoly_2]
    #vpoly_2c = [(WT_S[i],WT_PT[i]) for i in jpoly_2]
    ppoly_2c = Path(vpoly_2c)
    #ipoly_2 = np.where((ppoly_2c.contains_points(TScoords)) & (pres>=100.))[0]
    ipoly_2 = np.where((ppoly_2c.contains_points(TScoords)))[0] #INCLUDING SURFACE VALUES, NOT JUST LOWER THAN 100M

    #LA - polygon 3
    jpoly_3 = [3,10,9]
    vpoly_3 = [(WT_S[i],WT_PT[i]) for i in jpoly_3]
    vpoly_3c = [(WT_S[i]+corrS[i],WT_PT[i]+corrT[i]) for i in jpoly_3]
    #vpoly_3c = [(WT_S[i],WT_PT[i]) for i in jpoly_3]
    ppoly_3c = Path(vpoly_3c)
    #ipoly_3 = np.where((ppoly_3c.contains_points(TScoords)) & (pres>=100.))[0]
    ipoly_3 = np.where((ppoly_3c.contains_points(TScoords)))[0] #INCLUDING SURFACE VALUES, NOT JUST LOWER THAN 100M
    
    #LA - polygon 4
    jpoly_4 = [3,9,7]
    vpoly_4 = [(WT_S[i],WT_PT[i]) for i in jpoly_4]
    vpoly_4c = [(WT_S[i]+corrS[i],WT_PT[i]+corrT[i]) for i in jpoly_4]
    #vpoly_4c = [(WT_S[i],WT_PT[i]) for i in jpoly_4]
    ppoly_4c = Path(vpoly_4c)
    #ipoly_4 = np.where((ppoly_4c.contains_points(TScoords)) & (pres>=100.))[0]
    ipoly_4 = np.where((ppoly_4c.contains_points(TScoords)))[0] #INCLUDING SURFACE VALUES, NOT JUST LOWER THAN 100M
    
    #LA - polygon 5
    jpoly_5 = [9, 10, 8]
    vpoly_5 = [(WT_S[i],WT_PT[i]) for i in jpoly_5]
    vpoly_5c = [(WT_S[i]+corrS[i],WT_PT[i]+corrT[i]) for i in jpoly_5]
    #vpoly_5c = [(WT_S[i],WT_PT[i]) for i in jpoly_5]
    ppoly_5c = Path(vpoly_5c)
    #ipoly_5 = np.where((ppoly_5c.contains_points(TScoords)) & (pres>=100.))[0]
    ipoly_5 = np.where((ppoly_5c.contains_points(TScoords)))[0] #NCLUDING SURFACE VALUES, NOT JUST LOWER THAN 100M

    #LA - polygon 6
    jpoly_6 = [7,9,8]
    vpoly_6 = [(WT_S[i],WT_PT[i]) for i in jpoly_6]
    vpoly_6c = [(WT_S[i]+corrS[i],WT_PT[i]+corrT[i]) for i in jpoly_6]
    #vpoly_6c = [(WT_S[i],WT_PT[i]) for i in jpoly_6]
    ppoly_6c = Path(vpoly_6c)
    #ipoly_6 = np.where((ppoly_6c.contains_points(TScoords)) & (pres>=100.))[0]
    ipoly_6 = np.where((ppoly_6c.contains_points(TScoords)))[0] #INCLUDING SURFACE VALUES, NOT JUST LOWER THAN 100M

    
    #bottom waters - from Bieito's original code
    #jBW = [9,10]
    #iA= pDW.contains_points(TScoords)
    #iB = PT<3
    #iBW = np.where(~iA & iB)[0]
    
    #densidade TS #LA - changing the limits to fit SO
    ng = 25
    ti = np.linspace(-3,23,ng)
    si = np.linspace(33.25,36.5,ng)
    #ti = np.linspace(0,25,ng)
    #si = np.linspace(34.5,37.5,ng)
    deni = np.zeros((ng,ng))
    for i in range(ng):
        for j in range(ng):
            deni[i,j] = gsw.dens(si[j],ti[i],0)
    deni = deni - 1000.
    lvs = np.arange(deni.min(),deni.max(),0.5)
    
    #########
    ##OMPS##
    #######
    X = np.zeros((NP,NWM))
    #X[:] = np.nan
    #Preal = np.zeros((NP,1))
    #print Preal.shape
    print("Is doing OMP")
    
    #poly_1 
    print(PT[ipoly_1].shape)
    x=solve_OMP(PT,S,P,WT_PT,WT_S,WT_P,ipoly_1,jpoly_1)
    print(x.shape)
    X = realocates_values(x,X,ipoly_1,jpoly_1)
    #Preal = realocates_values(pres[iUCW].reshape((len(iUCW),1)),Preal,iUCW,[0])
    del x

    
    #poly_2
    x=solve_OMP(PT,S,P,WT_PT,WT_S,WT_P,ipoly_2,jpoly_2)
    X = realocates_values(x,X,ipoly_2,jpoly_2)
    del x
    
    #poly_3
    x=solve_OMP(PT,S,P,WT_PT,WT_S,WT_P,ipoly_3,jpoly_3)
    X = realocates_values(x,X,ipoly_3,jpoly_3)
    del x
    
    #poly_4
    x=solve_OMP(PT,S,P,WT_PT,WT_S,WT_P,ipoly_4,jpoly_4)
    X = realocates_values(x,X,ipoly_4,jpoly_4)
    del x

    #poly_5
    x=solve_OMP(PT,S,P,WT_PT,WT_S,WT_P,ipoly_5,jpoly_5)
    X = realocates_values(x,X,ipoly_5,jpoly_5)
    del x

    #poly_6
    x=solve_OMP(PT,S,P,WT_PT,WT_S,WT_P,ipoly_6,jpoly_6)
    X = realocates_values(x,X,ipoly_6,jpoly_6)
    del x

    
    #pon nans nos datos non resoltos
    ii = np.where(np.sum(X,axis=1)==0)[0]    
    X[ii,:] = np.nan
    #Preal[Preal==0] = np.nan
    
    print("OMP done!")

    wbWT =  pd.read_excel(WTfile)
    WT = wbWT['NAME'].tolist()
    WT_S = np.array(wbWT['S'].tolist())
    WT_PT = np.array(wbWT['T'].tolist())
    WT_P = np.array(wbWT['Ppre'].tolist()) 
    WT_dens = gsw.dens(WT_S,WT_PT,0)-1000
    
    WTo = {}
    WTo["WT"] = WT
    WTo["S"] = WT_S
    WTo["PT"] = WT_PT
    WTo["P"] = WT_P
    #WTo["NO"] = WT_NO #LA - don't need
    #WTo["Si"] = WT_Si #LA - don't need
    WTo["dens"] = WT_dens


    ##
    #Figures - only polygons (LA)
    ##
    
    f2, ax1 = plt.subplots(1, 1,figsize = (6,6))
    #polygons

    polypoly_1 = plt.Polygon(vpoly_1,alpha=0.2,fc='#014d4e',lw=0.5)
    ax1.add_patch(polypoly_1)

    polypoly_2 = plt.Polygon(vpoly_2,alpha=0.2,fc='#a2cffe',lw=0.5)
    ax1.add_patch(polypoly_2)
    
    polypoly_3 = plt.Polygon(vpoly_3,alpha=0.2,fc='#8e82fe',lw=0.5)
    ax1.add_patch(polypoly_3)
    
    polypoly_4 = plt.Polygon(vpoly_4,alpha=0.2,fc='aquamarine',lw=0.5)
    ax1.add_patch(polypoly_4)
    
    polypoly_5 = plt.Polygon(vpoly_5,alpha=0.2,fc='turquoise',lw=0.5)
    ax1.add_patch(polypoly_5)

    polypoly_6 = plt.Polygon(vpoly_6,alpha=0.2,fc='#cea2fd',lw=0.5)
    ax1.add_patch(polypoly_6)
    
    lvs = np.arange(deni.min(),deni.max(),0.5)
    CS=ax1.contour(si,ti,deni,levels=lvs,colors='gray')
    ax1.clabel(CS, fontsize=10, inline=1, fmt='%1.1f')
    ax1.scatter(S,PT,8,'gray',lw=0)
    ax1.scatter(WT_S,WT_PT,60,WT_dens, edgecolor = 'k', lw = 1, zorder =3, cmap = cm.jet)
    ax1.scatter(S,PT,8,'gray',lw=0)
    ax1.scatter(S,PT,8,'gray',lw=0)
    ax1.scatter(S[ipoly_1],PT[ipoly_1],8,'#014d4e',lw=0)
    ax1.scatter(S[ipoly_2],PT[ipoly_2],8,'#a2cffe',lw=0)
    ax1.scatter(S[ipoly_3],PT[ipoly_3],8,'#8e82fe',lw=0)
    ax1.scatter(S[ipoly_4],PT[ipoly_4],8,'aquamarine',lw=0)
    ax1.scatter(S[ipoly_5],PT[ipoly_5],8,'turquoise',lw=0)
    ax1.scatter(S[ipoly_6],PT[ipoly_6],8,'#cea2fd',lw=0)

    #ax1.scatter(S[iBW],PT[iBW],8,'k',lw=0)    
    
    ax1.set_xlim((33.25,36.35))
    ax1.set_ylim((-3,23))
    #ax1.set_ylim((-3,16))
    for i in range(len(WT)):
        ax1.annotate(WT[i],xy=(WT_S[i]+0.05,WT_PT[i]),va='center',fontsize=10)#,bbox = dict(fc='w',ec='w'))
    ax1.set_xlabel('S',fontsize = 16)
    ax1.set_ylabel('$\\theta$ ($^{\circ}$C)',fontsize=16)#polygons
    
    f2.savefig('TS_polygons_plot.png',bbox_inches = 'tight')
    plt.close(f2)

    return X, WTo

In [ ]:
def main():

    #LA - reading in MITgcm data
    ds_mar0_84_tave = xr.open_dataset(r"C:\Users\la1n23\OneDrive - University of Southampton\Documents\OCEAN_PHD\MITgcm\mar_0.84\5000_years\tave.0036720000.glob.nc")
    ds_mar0_84_ptr = xr.open_dataset(r"C:\Users\la1n23\OneDrive - University of Southampton\Documents\OCEAN_PHD\MITgcm\mar_0.84\5000_years\ptr_tave.0036720000.glob.nc")
    PT = ds_mar0_84_tave.Ttave.isel(T=-1).sel(Y = slice(-90,-30)).stack(all_lon_lats_depths=("X", "Y", "Z")).values
    S = ds_mar0_84_tave.Stave.isel(T=-1).sel(Y = slice(-90,-30)).stack(all_lon_lats_depths=("X", "Y", "Z")).values
    P = ds_mar0_84_ptr.ppre.isel(T=-1).sel(Y = slice(-90,-30)).stack(all_lon_lats_depths=("X", "Y", "Z")).values
    pres = abs((ds_mar0_84_tave.Stave.sel(Y = slice(-90,-30)).isel(T=-1).stack(all_lon_lats_depths=("X", "Y", "Z"))).Z.values)

    #does omp
    X, WT = does_OMP(pres,PT,S,P,r"C:\Users\la1n23\OneDrive - University of Southampton\Documents\OCEAN_PHD\Water_Masses_OMP\REDEFINED_WMTYPES.xlsx")

    Lat = (ds_mar0_84_tave.Ttave.isel(T=-1).sel(Y = slice(-90,-30)).stack(all_lon_lats_depths=("X", "Y", "Z"))).Y.values
    Lon = (ds_mar0_84_tave.Ttave.isel(T=-1).sel(Y = slice(-90,-30)).stack(all_lon_lats_depths=("X", "Y", "Z"))).X.values
    
    ##############
    #write output
    ###############

    #pickle
    out = {}
    out["Lat"] = Lat
    out["Lon"] = Lon
    out["WT"] = WT["WT"]
    out["X"] = X
    out["PT"] = PT
    out["S"] = S
    out["P"] = P
    out["pres"] = pres
    

    with open("Water Mass fractions.p", "wb" ) as f:
        pickle.dump( out , f ) 
    sio.savemat("Water Mass fractions",out)

    outVAR = ["Lon","Lat","pres","PT","S", "P"]
    nV = len(outVAR)
    
    wb = xlsxwriter.Workbook('Water Mass fractions.xlsx')
    sh1 = wb.add_worksheet()
    NWM = len(WT["WT"])
   # nD = len(out["Station"])
    nD = len(PT) 
    
    for cc in range(nV):
        var0 = outVAR[cc]
        sh1.write(0,cc,var0)
        for ff in range(nD):
            sh1.write(ff+1,cc,out[var0][ff])
    

    for i in range(NWM):
        sh1.write(0,i+nV+1,WT["WT"][i])
        for j in range(len(PT)):
            if np.isnan(X[j,i]) == False:
                sh1.write(j+1,i+nV+1,X[j,i])
    
    wb.close()

In [ ]:
main()